In [1]:
import pandas as pd
from Bio import Entrez
import time
# Define the file path
file_path = "../Datafile Raw/ov_tcga_pan_can_atlas_2018/data_mrna_seq_v2_rsem.txt"

# Load the data
df = pd.read_csv(file_path, sep="\t", low_memory=False)
df.head()



,Hugo_Symbol,Entrez_Gene_Id,TCGA-04-1348-01,TCGA-04-1357-01,TCGA-04-1362-01,TCGA-04-1364-01,TCGA-04-1365-01,TCGA-04-1514-01,TCGA-04-1519-01,TCGA-09-0364-01,...,TCGA-61-2102-01,TCGA-61-2104-01,TCGA-61-2109-01,TCGA-61-2110-01,TCGA-61-2111-01,TCGA-61-2113-01,TCGA-OY-A56P-01,TCGA-OY-A56Q-01,TCGA-VG-A8LO-01,TCGA-WR-A838-01
0,NaN,100130426,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,100133144,22.028266,17.671802,21.976720,14.940150,10.382058,2.733694,19.315232,20.896304,...,4.923308,20.466544,41.372838,20.519959,7.149914,21.566391,32.307387,26.118873,34.475700,126.819903
2,UBE2Q2P2,100134869,25.716221,20.473921,29.909980,7.827355,4.877826,6.227180,15.088948,27.821347,...,2.486953,29.214074,43.867435,48.929509,6.756215,8.630438,26.618719,15.716029,28.912917,59.400754
3,HMGB1P1,10357,321.241331,128.865135,424.021806,400.227877,891.927121,380.425710,507.815414,856.606003,...,335.265779,131.919622,305.443765,68.665161,225.455747,92.597754,203.931820,400.614342,309.174887,431.623999
4,NaN,10431,1952.678661,901.425735,1366.864827,1775.819439,1886.589299,1371.327702,1525.885484,721.340307,...,1596.217812,1128.424392,1026.626852,1172.472325,1142.036809,1291.281828,588.503145,1304.656225,1345.236051,1460.986221


In [2]:
# Count NaN values in 'Hugo_Symbol'
hugo_nan_count = df['Hugo_Symbol'].isna().sum()

# Count NaN values in 'Entrez_Gene_Id'
entrez_nan_count = df['Entrez_Gene_Id'].isna().sum()

print(f"Number of NaN values in 'Hugo_Symbol': {hugo_nan_count}")
print(f"Number of NaN values in 'Entrez_Gene_Id': {entrez_nan_count}")


Number of NaN values in 'Hugo_Symbol': 13
Number of NaN values in 'Entrez_Gene_Id': 0


In [3]:

# Set your email (required by NCBI Entrez)
Entrez.email = ""

def fetch_hugo_symbol(entrez_id):
    """
    Fetches the Hugo Symbol for a given Entrez Gene ID using the NCBI Entrez API.
    
    Parameters:
    entrez_id (str): The Entrez Gene ID to look up.
    
    Returns:
    str or None: The Hugo Symbol if found, otherwise None.
    """
    try:
        handle = Entrez.efetch(db="gene", id=str(entrez_id), rettype="xml")
        records = Entrez.parse(handle)

        hugo_symbol = None
        for record in records:
            gene_ref = record.get('Entrezgene_gene', {}).get('Gene-ref', {})
            hugo_symbol = gene_ref.get('Gene-ref_locus') or gene_ref.get('Gene-ref_locus-tag')

        return hugo_symbol
    except Exception as e:
        print(f"Error fetching Entrez_Gene_Id {entrez_id}: {e}")
        return None

# Find rows where 'Hugo_Symbol' is missing
missing_symbols = df['Hugo_Symbol'].isna()
missing_count = missing_symbols.sum()
print(f"Number of missing Hugo_Symbol values: {missing_count}")

# Iterate and update missing values
for idx, row in df[missing_symbols].iterrows():
    entrez_id = row['Entrez_Gene_Id']
    
    if pd.notna(entrez_id):  # Ensure Entrez ID is valid
        hugo_symbol = fetch_hugo_symbol(entrez_id)
        if hugo_symbol:
            df.at[idx, 'Hugo_Symbol'] = hugo_symbol
            print(f"Updated row {idx}: Entrez_Gene_Id {entrez_id} -> Hugo_Symbol {hugo_symbol}")
        else:
            print(f"Hugo_Symbol not found for Entrez_Gene_Id {entrez_id}")
    
    # Respect NCBI API rate limit (max 3 requests per second)
    time.sleep(0.35)


Number of missing Hugo_Symbol values: 13
Updated row 0: Entrez_Gene_Id 100130426 -> Hugo_Symbol LOC100130426
Updated row 1: Entrez_Gene_Id 100133144 -> Hugo_Symbol UBE2Q2P3
Updated row 4: Entrez_Gene_Id 10431 -> Hugo_Symbol TIMM23
Updated row 5: Entrez_Gene_Id 136542 -> Hugo_Symbol MOXD2
Updated row 6: Entrez_Gene_Id 155060 -> Hugo_Symbol LOC155060
Updated row 9: Entrez_Gene_Id 317712 -> Hugo_Symbol LOC317712
Updated row 13: Entrez_Gene_Id 391343 -> Hugo_Symbol LOC391343
Updated row 18: Entrez_Gene_Id 553137 -> Hugo_Symbol LOC553137
Updated row 19: Entrez_Gene_Id 57714 -> Hugo_Symbol KIAA1618
Updated row 20: Entrez_Gene_Id 645851 -> Hugo_Symbol LOC645851
Updated row 21: Entrez_Gene_Id 652919 -> Hugo_Symbol RGPD7
Updated row 24: Entrez_Gene_Id 728603 -> Hugo_Symbol FRMPD2P2
Updated row 26: Entrez_Gene_Id 729884 -> Hugo_Symbol TMPRSS11E2


In [4]:
# Count NaN values in 'Hugo_Symbol'
hugo_nan_count = df['Hugo_Symbol'].isna().sum()

# Count NaN values in 'Entrez_Gene_Id'
entrez_nan_count = df['Entrez_Gene_Id'].isna().sum()

print(f"Number of NaN values in 'Hugo_Symbol': {hugo_nan_count}")
print(f"Number of NaN values in 'Entrez_Gene_Id': {entrez_nan_count}")


Number of NaN values in 'Hugo_Symbol': 0
Number of NaN values in 'Entrez_Gene_Id': 0


In [5]:
df

,Hugo_Symbol,Entrez_Gene_Id,TCGA-04-1348-01,TCGA-04-1357-01,TCGA-04-1362-01,TCGA-04-1364-01,TCGA-04-1365-01,TCGA-04-1514-01,TCGA-04-1519-01,TCGA-09-0364-01,...,TCGA-61-2102-01,TCGA-61-2104-01,TCGA-61-2109-01,TCGA-61-2110-01,TCGA-61-2111-01,TCGA-61-2113-01,TCGA-OY-A56P-01,TCGA-OY-A56Q-01,TCGA-VG-A8LO-01,TCGA-WR-A838-01
0,LOC100130426,100130426,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,UBE2Q2P3,100133144,22.028266,17.671802,21.976720,14.940150,10.382058,2.733694,19.315232,20.896304,...,4.923308,20.466544,41.372838,20.519959,7.149914,21.566391,32.307387,26.118873,34.475700,126.819903
2,UBE2Q2P2,100134869,25.716221,20.473921,29.909980,7.827355,4.877826,6.227180,15.088948,27.821347,...,2.486953,29.214074,43.867435,48.929509,6.756215,8.630438,26.618719,15.716029,28.912917,59.400754
3,HMGB1P1,10357,321.241331,128.865135,424.021806,400.227877,891.927121,380.425710,507.815414,856.606003,...,335.265779,131.919622,305.443765,68.665161,225.455747,92.597754,203.931820,400.614342,309.174887,431.623999
4,TIMM23,10431,1952.678661,901.425735,1366.864827,1775.819439,1886.589299,1371.327702,1525.885484,721.340307,...,1596.217812,1128.424392,1026.626852,1172.472325,1142.036809,1291.281828,588.503145,1304.656225,1345.236051,1460.986221
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20526,ZYG11A,440590,6.824650,4.775693,66.800880,15.651470,8.691137,24.236724,3.320645,10.812258,...,10.768112,37.768199,11.964642,23.172043,84.289826,137.984157,33.829916,141.004912,7.760542,79.382924
20527,ZYG11B,79699,483.942790,518.654716,791.336454,701.294430,891.615830,935.538347,1081.073819,410.169901,...,714.170417,890.967623,913.879586,934.138220,860.248654,670.929813,1793.289275,1000.073852,1321.299392,1068.161532
20528,ZYX,7791,9692.870954,3284.754545,3747.937644,3647.207428,4404.299470,3583.030316,9961.232422,2941.540137,...,11158.354211,4559.453197,5640.258550,9311.826985,2797.563049,4145.227437,6202.950375,6698.448007,4961.305257,3907.580348
20529,ZZEF1,23140,429.818654,684.474498,766.249800,316.699864,436.880502,752.320778,428.876734,827.360674,...,400.605537,934.263473,595.691754,547.418363,485.261856,1175.311636,1291.699942,448.557587,709.015420,1665.424483


In [6]:
# Identify patient columns (excluding 'Hugo_Symbol' and 'Entrez_Gene_Id')
patient_columns = df.columns[2:]  # Excluding metadata columns

# Calculate the percentage of NaN values for each gene
nan_percentage = df[patient_columns].isna().sum(axis=1) / len(patient_columns)

# Count the number of genes with 80% or more NaN values
num_genes_with_80_percent_nan = (nan_percentage >= 0.8).sum()

# Calculate the total number of genes in the dataset
total_genes = df.shape[0]

# Compute the percentage of affected genes
percentage_of_genes = (num_genes_with_80_percent_nan / total_genes) * 100

# Print the results
print(f"Total number of genes: {total_genes}")
print(f"Number of genes with 80% or more NaN values: {num_genes_with_80_percent_nan}")
print(f"Percentage of genes with 80% or more NaN values: {percentage_of_genes:.2f}%")


Total number of genes: 20531
Number of genes with 80% or more NaN values: 1455
Percentage of genes with 80% or more NaN values: 7.09%


In [7]:
# Identify patient columns (excluding 'Hugo_Symbol' and 'Entrez_Gene_Id')
patient_columns = df.columns[2:]  # Excluding metadata columns

# Calculate the percentage of NaN values for each gene
nan_percentage = df[patient_columns].isna().sum(axis=1) / len(patient_columns)

# Filter out rows where 80% or more of the values are NaN
df = df.loc[nan_percentage < 0.8].reset_index(drop=True)

# Print confirmation message
print(f"Updated dataset: {df.shape[0]} genes remaining after removing {nan_percentage.ge(0.8).sum()} genes.")


Updated dataset: 19076 genes remaining after removing 1455 genes.


In [8]:
df=df.transpose()
df.loc["Hugo_Symbol"] = df.loc["Hugo_Symbol"].map(str)

In [9]:
df

,0,1,2,3,4,5,6,7,8,9,...,19066,19067,19068,19069,19070,19071,19072,19073,19074,19075
Hugo_Symbol,UBE2Q2P3,UBE2Q2P2,HMGB1P1,TIMM23,LOC155060,RNU12-2P,EZHIP,EFCAB8,SRP14P1,TRIM75P,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
Entrez_Gene_Id,100133144,100134869,10357,10431,155060,26823,340602,388795,390284,391714,...,55055,11130,7789,158586,79364,440590,79699,7791,23140,26009
TCGA-04-1348-01,22.028266,25.716221,321.241331,1952.678661,222.150613,0.676632,-0.059851,3.038671,4.594825,-0.329284,...,1018.226589,1181.934902,25.685296,189.275552,784.054335,6.82465,483.94279,9692.870954,429.818654,475.233973
TCGA-04-1357-01,17.671802,20.473921,128.865135,901.425735,244.41232,0.09028,-0.479632,4.932117,5.20536,-0.329284,...,213.949506,323.811661,79.007175,469.565537,1177.56175,4.775693,518.654716,3284.754545,684.474498,414.753874
TCGA-04-1362-01,21.97672,29.90998,424.021806,1366.864827,377.449014,0.451162,4.514755,3.434062,4.371905,0.780617,...,462.214865,900.545359,66.976049,634.928214,1493.09879,66.80088,791.336454,3747.937644,766.2498,802.717598
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-61-2113-01,21.566391,8.630438,92.597754,1291.281828,439.904885,4.662041,-0.297816,2.292796,4.521237,1.076939,...,592.350119,681.65494,16.591836,128.124814,1210.742246,137.984157,670.929813,4145.227437,1175.311636,970.703672
TCGA-OY-A56P-01,32.307387,26.618719,203.93182,588.503145,510.841037,1.546676,1.883775,1.74242,2.415765,0.208295,...,356.505317,451.362926,71.339022,513.147769,2245.23428,33.829916,1793.289275,6202.950375,1291.699942,2822.170162
TCGA-OY-A56Q-01,26.118873,15.716029,400.614342,1304.656225,387.776781,0.453561,-0.306245,2.8211,1.695522,1.011678,...,429.84272,930.484213,35.019082,239.974253,1757.926551,141.004912,1000.073852,6698.448007,448.557587,2462.888278
TCGA-VG-A8LO-01,34.4757,28.912917,309.174887,1345.236051,412.614342,7.464714,0.693589,1.802044,6.904617,1.831025,...,903.788016,896.533687,51.943411,396.134723,1816.741351,7.760542,1321.299392,4961.305257,709.01542,1295.464299


In [10]:
# Define the file path
file_path = "../Datafiles Final/Datafiles Cleaned/mrnaseqv2.pkl"

# Save using Pickle format (best for mixed data types)
df.to_pickle(file_path)

print(f"File saved successfully at: {file_path}")


File saved successfully at: ../Datafiles Final/Datafiles Cleaned/mrnaseqv2.pkl


In [11]:
# Load the Pickle file
df_mrnaseq = pd.read_pickle(file_path)

# Verify the data
df_mrnaseq

,0,1,2,3,4,5,6,7,8,9,...,19066,19067,19068,19069,19070,19071,19072,19073,19074,19075
Hugo_Symbol,UBE2Q2P3,UBE2Q2P2,HMGB1P1,TIMM23,LOC155060,RNU12-2P,EZHIP,EFCAB8,SRP14P1,TRIM75P,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
Entrez_Gene_Id,100133144,100134869,10357,10431,155060,26823,340602,388795,390284,391714,...,55055,11130,7789,158586,79364,440590,79699,7791,23140,26009
TCGA-04-1348-01,22.028266,25.716221,321.241331,1952.678661,222.150613,0.676632,-0.059851,3.038671,4.594825,-0.329284,...,1018.226589,1181.934902,25.685296,189.275552,784.054335,6.82465,483.94279,9692.870954,429.818654,475.233973
TCGA-04-1357-01,17.671802,20.473921,128.865135,901.425735,244.41232,0.09028,-0.479632,4.932117,5.20536,-0.329284,...,213.949506,323.811661,79.007175,469.565537,1177.56175,4.775693,518.654716,3284.754545,684.474498,414.753874
TCGA-04-1362-01,21.97672,29.90998,424.021806,1366.864827,377.449014,0.451162,4.514755,3.434062,4.371905,0.780617,...,462.214865,900.545359,66.976049,634.928214,1493.09879,66.80088,791.336454,3747.937644,766.2498,802.717598
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-61-2113-01,21.566391,8.630438,92.597754,1291.281828,439.904885,4.662041,-0.297816,2.292796,4.521237,1.076939,...,592.350119,681.65494,16.591836,128.124814,1210.742246,137.984157,670.929813,4145.227437,1175.311636,970.703672
TCGA-OY-A56P-01,32.307387,26.618719,203.93182,588.503145,510.841037,1.546676,1.883775,1.74242,2.415765,0.208295,...,356.505317,451.362926,71.339022,513.147769,2245.23428,33.829916,1793.289275,6202.950375,1291.699942,2822.170162
TCGA-OY-A56Q-01,26.118873,15.716029,400.614342,1304.656225,387.776781,0.453561,-0.306245,2.8211,1.695522,1.011678,...,429.84272,930.484213,35.019082,239.974253,1757.926551,141.004912,1000.073852,6698.448007,448.557587,2462.888278
TCGA-VG-A8LO-01,34.4757,28.912917,309.174887,1345.236051,412.614342,7.464714,0.693589,1.802044,6.904617,1.831025,...,903.788016,896.533687,51.943411,396.134723,1816.741351,7.760542,1321.299392,4961.305257,709.01542,1295.464299


In [12]:
#####################################################################################################